# Sahformer — Kaggle PONDER training (big, resumable)

Trains the **pondering** model (adaptive computation + clock-gated halting) on a large streamed
dataset, and **resumes across sessions** so you can do it in chunks.

**Settings panel (right):**
- **Accelerator → GPU (T4)**
- **Internet → On**
- **Persistence → Files only**  ← *required*: keeps your data shards + checkpoint between
  sessions so resuming works.

**How resuming works:** run it, let it train, session ends. Next session, run top-to-bottom again
— it **skips rebuilding the data** (shards persist) and **resumes from the last checkpoint**
automatically. Repeat until `max_steps` is reached.

**Honest note on size:** Kaggle's working disk is ~20 GB, so ~100M positions **will not fit** as
shards (they'd be tens of GB). `MAX_POSITIONS` below defaults to a big-but-feasible **20M**. If the
data cell runs out of disk, lower it. (True 100M-scale needs more disk than free Kaggle offers.)

**On time:** uses a **~14M model** (trimmed for speed) with `grad_clip` on to prevent NaN
blow-ups. Pondering still does extra work per step, so it's a **multi-hour, resumable** run — do
it as **Save & Run All (Commit)** and resume across sessions. Raise `dim_vit`/`num_blocks` for a
bigger model only if you're ready for much longer runs.

In [ ]:
# 1) Get the code
REPO_URL = "https://github.com/slobaspid/sah-transformer.git"  # <-- EDIT if different
import os
if not os.path.isdir("sah-transformer"):
    !git clone "$REPO_URL"
%cd sah-transformer
!git pull --ff-only || true

In [ ]:
# 2) Deps + GPU check
!pip -q install python-chess zstandard
import sys; sys.path.insert(0, ".")
import torch
ok = torch.cuda.is_available()
print("torch", torch.__version__, "| cuda", ok, "|",
      torch.cuda.get_device_name(0) if ok else "NO GPU -> Settings > Accelerator > GPU")

## 3) Build (or reuse) the data shards

Streams a 2017-04+ month, chunked so memory stays flat. **Skips the build if shards already
exist** (so resuming a later session is fast). Watch the disk — lower `MAX_POSITIONS` if it fills.

In [ ]:
import glob
from sahformer.download import stream_games_from_url
from sahformer.dataset_build import build_shards, records_from_games

URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2017-04.pgn.zst"
MAX_POSITIONS = 20_000_000   # big-but-feasible on Kaggle's ~20GB disk; lower if disk fills
BALANCE = False

DATA_SHARDS = sorted(glob.glob("data/*.npz"))
if DATA_SHARDS:
    print("reusing", len(DATA_SHARDS), "existing shards (skipping build)")
else:
    build_shards(records_from_games(stream_games_from_url(URL)),
                 "data", chunk_positions=250000, max_positions=MAX_POSITIONS,
                 balance=BALANCE, progress_every=50000)
    DATA_SHARDS = sorted(glob.glob("data/*.npz"))
print(len(DATA_SHARDS), "shards ready")

In [ ]:
OUT = "/kaggle/working/sahformer_ckpts"
import os; os.makedirs(f"{OUT}/ponder", exist_ok=True)
print("checkpoints ->", OUT)

## 4) Train the ponder model (resumable, with anti-collapse failsafes)

`max_steps` is the **total** target across all sessions; each run trains toward it and
**auto-resumes** from `last.pt`. Failsafes keep the halting from collapsing to 1 step:
- `ponder_warmup` — first N steps train *all* ponder steps equally so the block learns to refine.
- `ponder_min_steps` + `ponder_floor_beta` — a *gentle* floor so it stays above ~1.5 steps while
  still being adaptive. Watch the `avg ponder steps` plot: it should sit in the ~2–4 range, not
  pin to 1 (collapsed) or 4 (always-max). If it collapses, raise `ponder_floor_beta`; if it pins
  at 4, lower it.

In [ ]:
import os, glob, torch
from sahformer.model.config import ModelConfig
from sahformer.training.build import build_model
from sahformer.training.loop import TrainConfig, train
DATA_SHARDS = sorted(glob.glob("data/*.npz"))

MODEL = ModelConfig(dim_vit=384, num_blocks=10)   # ~14M backbone + ponder/halt heads (finishable)
print(sum(p.numel() for p in build_model("ponder", MODEL).parameters())/1e6, "M params")

def _clean_ckpt(path):
    if not os.path.exists(path):
        return False
    try:
        sd = torch.load(path, map_location="cpu", weights_only=False)["model_state"]
        return all(torch.isfinite(v).all() for v in sd.values() if v.is_floating_point())
    except Exception:
        return False

resume = ""
for cand in (f"{OUT}/ponder/last.pt", f"{OUT}/ponder/best.pt"):   # prefer latest clean checkpoint
    if _clean_ckpt(cand):
        resume = cand; break
print("RESUMING from", resume) if resume else print("fresh start (no clean checkpoint)")

cfg = TrainConfig(mode="ponder", max_steps=80000, warmup_steps=2000, batch_size=512,
                  lr=3e-4, amp=True, stream=True, resume=resume,
                  ponder_beta=0.02, ponder_warmup=2000,        # <- anti-collapse failsafes
                  ponder_min_steps=1.5, ponder_floor_beta=0.15,
                  device="cuda", out_dir=f"{OUT}/ponder", log_every=200, ckpt_every=1000)
res = train(cfg, DATA_SHARDS, model_cfg=MODEL)
print("ponder best_total:", res["best"])

In [ ]:
import matplotlib.pyplot as plt
h = res["history"]; xs = [r["step"] for r in h]
plt.plot(xs, [r["total"] for r in h]); plt.title("ponder — total loss")
plt.xlabel("step"); plt.show()
plt.plot(xs, [r["time"] for r in h]); plt.title("avg ponder steps (should sit >1, not collapse)")
plt.xlabel("step"); plt.ylabel("avg steps"); plt.show()

## 4b) Poke it — does it think DEEPER on hard positions & LESS under time pressure?

Feeds a sparse/easy position and a dense/sharp one, and prints how many ponder steps it used for
each. Then the same hard position with lots of clock vs almost none. If pondering learned what we
hoped: **hard ≥ easy**, and **low clock < high clock**.

In [ ]:
import chess, torch
from sahformer.encoding import encode_board, build_temporal
from sahformer.records import _stack_history
model = res["model"]; model.eval()
dev = next(model.parameters()).device

def ponder_steps_for(fen, clock=120.0):
    b = chess.Board(fen)
    cur = encode_board(b); hist = _stack_history([], cur)
    tmp = build_temporal(clock, clock, [], 20)
    batch = {"board": torch.from_numpy(cur).float().unsqueeze(0).to(dev),
             "history": torch.from_numpy(hist).float().unsqueeze(0).to(dev),
             "elo_self": torch.tensor([2000]).to(dev), "elo_opp": torch.tensor([2000]).to(dev),
             "temporal": torch.from_numpy(tmp).float().unsqueeze(0).to(dev)}
    return model(batch)["ponder_steps"]

EASY = "8/5k2/8/8/8/8/5K2/6R1 w - - 0 1"                                    # sparse, simple
HARD = "r2q1rk1/1b1nbppp/p2ppn2/1p6/3NPP2/2N1B3/PPPQB1PP/R4RK1 w - - 0 1"   # dense, sharp
print("EASY position  -> ponder_steps:", ponder_steps_for(EASY))
print("HARD position  -> ponder_steps:", ponder_steps_for(HARD))
print("HARD @ 120s clock ->", ponder_steps_for(HARD, 120.0),
      " | HARD @ 2s clock ->", ponder_steps_for(HARD, 2.0))

## 5) SAVE & DOWNLOAD your model  (do this before the session ends!)

In [ ]:
import os, shutil
CKPT_DIR = f"{OUT}/ponder"
for f in sorted(os.listdir(CKPT_DIR)):
    p = os.path.join(CKPT_DIR, f); print(f"   {p}   ({os.path.getsize(p)/1e6:.1f} MB)")
shutil.make_archive("/kaggle/working/model_ponder", "zip", CKPT_DIR)
print("\n==> DOWNLOAD: /kaggle/working/model_ponder.zip  (Output panel -> Download)")
print("    Persistence=Files only keeps it on Kaggle so the next session resumes automatically.")

## Done

Checkpoints are in `/kaggle/working/sahformer_ckpts/ponder/`. To continue later: reopen with the
same Persistence, run top to bottom — it reuses the shards and resumes from `last.pt`.

The checkpoint is self-describing, so your local viewer/engine load the ponder model
automatically. In self-play, `out["ponder_steps"]` tells you how deep it thought on each move.